# 03 - Evaluate

Scores each system's alignments against the beat annotations and reproduces the error
rate table and figures.

Scoring itself is done by `benchmark.py evaluate`, which writes one `errs.pkl` per
system into the benchmark's eval directory. This notebook loads those files and
summarises them.

In [ ]:
%matplotlib inline

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from corpora.benchmarks import BENCHMARK_CONFIGS
from utils.constants import display_name

## Configuration

Systems are named by their experiment-directory keys; `display_name` maps them to the
labels used in the paper. Keys not in the map are shown as-is, so parameter sweeps
such as `OLTW_GLOBAL_B` can be dropped into this list unchanged.

In [ ]:
BENCHMARK = "test"
EVAL_DIR = BENCHMARK_CONFIGS[BENCHMARK]["eval_dir"]

systems = ["DTW", "OLTW", "MM_ARZT", "MM_DIXON", "OLTW_OURS", "OLTW_GLOBAL", "SOA", "SOA_MONOTONIC"]
tols = [50, 100, 200, 500, 1000, 2000]  # error tolerances in milliseconds

## Score the alignments

Evaluates every system directory found under the benchmark's experiments directory.
Skip this cell if `benchmark.py evaluate` (or `benchmark.py run`) has already been run.

In [ ]:
!python benchmark.py evaluate --benchmark {BENCHMARK}

## Error rate vs. tolerance

In [ ]:
def load_error_rates(eval_dir, system, tols):
    '''Fraction of annotated beats aligned outside each error tolerance.

    Inputs
    eval_dir: directory holding the per-system eval output
    system: system key, i.e. the name of the subdirectory holding errs.pkl
    tols: error tolerances in milliseconds

    Returns an array of error rates, one per tolerance.
    '''
    with open(f'{eval_dir}/{system}/errs.pkl', 'rb') as f:
        d = pickle.load(f)

    errs = np.concatenate([d[scenario_id] for scenario_id in d]) if d else np.array([])
    errs = errs[~np.isnan(errs)]
    return np.array([np.mean(np.abs(errs) > tol / 1000) for tol in tols])

In [ ]:
def plotErrorVsTolerance(eval_dir, systems, tols, savefile=None, style='bar'):
    '''Plots the error rate across a range of error tolerances.

    Inputs
    eval_dir: directory holding the per-system eval output
    systems: system keys to plot
    tols: error tolerances in milliseconds
    savefile: if specified, saves the figure to the given filepath
    style: 'bar' or 'line'

    Returns a DataFrame of error rates, indexed by tolerance.
    '''
    errRates = {system: load_error_rates(eval_dir, system, tols) for system in systems}

    bar_width = 0.8 / len(systems)
    for i, system in enumerate(systems):
        if style == 'line':
            plt.plot(tols, errRates[system] * 100.0, label=display_name(system))
        else:
            pos = np.arange(len(tols)) + i * bar_width
            plt.bar(pos, errRates[system] * 100.0, width=bar_width, label=display_name(system))

    if style != 'line':
        plt.xticks(np.arange(len(tols)) + 0.4 - bar_width / 2, map(str, tols))
    plt.ylabel('Error Rate (%)')
    plt.xlabel('Error Tolerance (ms)')
    plt.legend()
    plt.grid(linestyle='--')
    if savefile:
        os.makedirs(os.path.dirname(savefile), exist_ok=True)
        plt.savefig(savefile, bbox_inches='tight', dpi=200)
    plt.show()

    df = pd.DataFrame({display_name(s): errRates[s] for s in systems}, index=tols)
    df.index.name = 'Tolerance (ms)'
    df.columns.name = 'System'
    return df

Error rate curve for a single system:

In [ ]:
_ = plotErrorVsTolerance(EVAL_DIR, ['SOA'], tols, style='line')

All systems compared:

In [ ]:
df = plotErrorVsTolerance(EVAL_DIR, systems, tols, savefile='figures/error_rate_vs_tolerance.png')
display(df)

## Export the results table

`results/error_rates_<benchmark>.csv` is the aggregate the paper's tables are built
from. It is small enough to keep under version control, unlike the raw alignments.

In [ ]:
os.makedirs('results', exist_ok=True)
outfile = f'results/error_rates_{BENCHMARK}.csv'
df.to_csv(outfile)
print(f'wrote {outfile}')